# Notebook de criação + ingestão de dados

Criação do catálogo, schemas de cada camada e volume para armazenar os CSVs

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce_lakehouse;
CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.bronze;
CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.silver;
CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.gold;

CREATE VOLUME IF NOT EXISTS ecommerce_lakehouse.bronze.rawData;

Criação das tabelas na camada bronze

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

volume_path = "/Volumes/ecommerce_lakehouse/bronze/rawdata/dataset/"

csv_files = dbutils.fs.ls(volume_path)

for file_info in csv_files:
    if file_info.name.endswith('.csv'):
        table_name = file_info.name.replace('.csv', '')
        table_name = table_name.replace('_dataset', '')
        table_name = table_name.replace('olist_', '')
        table_name = 'tb_' + table_name
        
        # Ler CSV
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(file_info.path) \
            .withColumn("timestamp_ingestion", F.current_timestamp())
        
        # Salvar como Delta Table na camada Bronze
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"ecommerce_lakehouse.bronze.{table_name}")
        
        print(f"Tabela bronze.{table_name} criada com {df.count()} registros")

✓ Tabela bronze.tb_customers criada com 99441 registros
✓ Tabela bronze.tb_geolocation criada com 1000163 registros
✓ Tabela bronze.tb_order_items criada com 112650 registros
✓ Tabela bronze.tb_order_payments criada com 103886 registros
✓ Tabela bronze.tb_order_reviews criada com 104162 registros
✓ Tabela bronze.tb_orders criada com 99441 registros
✓ Tabela bronze.tb_products criada com 32951 registros
✓ Tabela bronze.tb_sellers criada com 3095 registros
✓ Tabela bronze.tb_product_category_name_translation criada com 71 registros


Criação da tabela de ingestão da API do BaCen

In [0]:
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import datetime

df_pedidos = spark.read.table("ecommerce_lakehouse.bronze.tb_orders")

df_datas_limites = df_pedidos.agg(
    F.min("order_purchase_timestamp").alias("data_inicio"),
    F.max("order_purchase_timestamp").alias("data_fim")
)

data_inicio = df_datas_limites.first()["data_inicio"].strftime("%m-%d-%Y")
data_fim    = df_datas_limites.first()["data_fim"].strftime("%m-%d-%Y")

print(f"Data início: {data_inicio}")
print(f"Data fim: {data_fim}")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    f"&$select=dataHoraCotacao,cotacaoCompra"
    f"&$format=json"
)

print(f"Consultando API...")
response = requests.get(url, timeout=30)

if response.status_code != 200:
    raise Exception(f"Erro na API: HTTP {response.status_code} -> {response.text}")

registros = response.json().get("value", [])

if not registros:
    raise Exception("Nenhuma cotação retornada para o período informado. Verifique as datas (apenas dias úteis são retornados).")

print(f"{len(registros)} cotações recebidas da API")

schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra",   DoubleType(), True),
])

df_cotacao = spark.createDataFrame(registros, schema=schema)

df_cotacao = (
    df_cotacao
    # Converte string -> timestamp -> extrai só a data
    .withColumn("dataHoraCotacao", F.to_timestamp("dataHoraCotacao", "yyyy-MM-dd HH:mm:ss.SSS"))
    .withColumn("dataCotacao",     F.to_date("dataHoraCotacao"))
    .withColumnRenamed("cotacaoCompra", "vlCotacaoCompra")
    # Metadados de auditoria
    .withColumn("ingested_timestamp",  F.current_timestamp())
    .withColumn("source",       F.lit("API BaCen"))
    .withColumn("param_inicio", F.lit(data_inicio))
    .withColumn("param_fim",    F.lit(data_fim))
)

df_cotacao.display()

table_name = "ecommerce_lakehouse.bronze.tb_cotacao_dolar"

df_cotacao.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"Tabela '{table_name}' salva com {df_cotacao.count()} registros.")

Data início: 09-04-2016
Data fim: 10-17-2018
Consultando API...
530 cotações recebidas da API


dataHoraCotacao,vlCotacaoCompra,dataCotacao,ingested_timestamp,source,param_inicio,param_fim
2016-09-05T13:09:55.659Z,3.2715,2016-09-05,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-06T13:02:39.984Z,3.2446,2016-09-06,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-08T13:03:53.968Z,3.1928,2016-09-08,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-09T13:14:00.885Z,3.2632,2016-09-09,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-12T13:08:01.541Z,3.2848,2016-09-12,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-13T13:03:56.534Z,3.2966,2016-09-13,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-14T13:05:51.819Z,3.3256,2016-09-14,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-15T13:08:34.825Z,3.332,2016-09-15,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-16T13:04:11.365Z,3.2998,2016-09-16,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018
2016-09-19T13:07:42.039Z,3.263,2016-09-19,2026-04-07T19:16:58.498Z,API BaCen,09-04-2016,10-17-2018


Tabela 'ecommerce_lakehouse.bronze.tb_cotacao_dolar' salva com 530 registros.
